In [6]:
import cv2
import pandas as pd
import numpy as np
import pickle
from collections import Counter

In [ ]:

# Import data
DATA_ROOT = "C:/Users/marko/Desktop/NAIST-UBI-RESEARCH/FSAR_Team_Sports/multisports_data/data/trainval"
MEDATATA_PATH = DATA_ROOT + "/multisports_GT.pkl"

with open(MEDATATA_PATH, "rb") as file:
    data = pickle.load(file)

labels = data["labels"]
gttubes = data["gttubes"]

# Get class counts
class_counts = Counter()
for vid, actions in gttubes.items():
    for label_id, tubes in actions.items():
        class_counts[label_id] += len(tubes)

# First remove the lowest 6 classes cause there isn't enough samples to do anything 
remove_ids = [cid for cid, count in class_counts.items() if count < 25]
for vid_key, vid_tublets in gttubes.items():
    for lab_id in list(vid_tublets.keys()):
        if lab_id in remove_ids:
            del gttubes[vid_key][lab_id]

    if not gttubes[vid_key]:
        print("here")
        del gttubes[vid_key]

# Adjust the counter
class_counts = Counter()
for vid, actions in gttubes.items():
    for label_id, tubes in actions.items():
        class_counts[label_id] += len(tubes)


# Then use the N lowest count classes from the leftover classes for the test set
N = 15
sorted_by_count = sorted(class_counts.items(), key=lambda item: item[1])
test_ids = sorted_by_count[:N]
test_ids = [item[0] for item in test_ids]

# For counting number of test and train samples
train_sample_counter = 0
test_sample_counter = 0

# Get all the videos with these classes and put them in the test class and the other videos in the train class
test_videos = []
train_videos = []
for vid_key, vid_tublets in gttubes.items():
    possible_train = True
    for lab_id in list(vid_tublets.keys()):
        if lab_id in test_ids:
            test_videos.append(vid_key)
            possible_train = False
            test_sample_counter += sum(len(tubes) for lab_id, tubes in vid_tublets.items() if lab_id in test_ids)
            break

    if possible_train:
        train_videos.append(vid_key)
        train_sample_counter += sum(len(tubes) for tubes in vid_tublets.values())

# Count the number of test videos and train videos
print(f"Number of test videos: {len(test_videos)}")
print(f"Number of train videos: {len(train_videos)}")

# Number of train and test samples
print(f"Number of train instances: {train_sample_counter}")
print(f"Number of test instances: {test_sample_counter}")


# Re-indexing
remaining_ids = sorted(class_counts.keys())
label_map = {old_id: new_id for new_id, old_id in enumerate(remaining_ids)}
processed_gttubes = {}

for vid_key, vid_tublets in gttubes.items():
    processed_gttubes[vid_key] = {label_map[old_id]: tubes for old_id, tubes in vid_tublets.items()}

new_labels = [labels[i] for i in remaining_ids]
new_test_ids = [label_map[tid] for tid in test_ids]

# Save the splits back into a new pickle file
split_dict = {
    "labels": new_labels,
    "train_videos": train_videos,
    "test_videos": test_videos,
    "nframes": data["nframes"],
    "resolution": data["resolution"],
    "gttubes": gttubes
}

# Create the ground truth pickle file
with open("multisports_fewshot_GT.pkl", "wb") as f:
    pickle.dump(split_dict, f)
